# TP 08 | Peut-on mettre une règle d’inspection en service ?

**Chapitre 5. Mettre le modèle en production**

Camille doit décider comment ouvrir la file de contrôle des panneaux chez Nervure. Deux règles ont été livrées : le pipeline local XGBoost du TP 05 et la régression logistique Spark du TP 07. Votre décision : peut-on ouvrir automatiquement la file sans dépasser les 48 contrôles disponibles ?

**Avant de commencer.** Terminez les TP 05 et 07, ou exécutez leurs corrigés dans ce dépôt. Ils créent vos propres versions sous `artefacts/tp05/` et `artefacts/tp07/`. Dans VS Code, ouvrez le dossier du dépôt, exécutez `uv sync --locked --extra spark` et sélectionnez le noyau du projet. Le premier démarrage de Java requiert Internet.

**À remettre.** Une fiche de décision en fin de notebook : mode de service, version de chaque règle, seuil, charge, coût, rappel, risques et conduite à tenir. Les trois défis facultatifs s’appuient sur le même journal de décisions. Gardez le fichier de verdicts fermé jusqu’à la section 5.

## 1. Choisir comment livrer une priorité

**Atelier 8.1 du support MLB.** Sections 1 à 3.

Camille décrit trois situations. Pour chacune, choisissez le mode qui respecte le délai et expliquez ce qui se passe si le service est indisponible.

| Situation à l’usine | Mode à envisager | Point à vérifier |
| --- | --- | --- |
| Préparer la liste du lendemain | **Batch** : un calcul planifié sur les lots reçus | À quelle heure la liste doit-elle être prête ? |
| Afficher les nouvelles priorités sous cinq minutes | **Micro-batch** : traiter régulièrement les dernières arrivées | Combien de temps un lot attend-il la prochaine collecte ? |
| Renseigner un opérateur qui interroge un lot précis | **Endpoint synchrone** : l’application attend la réponse du service | Que voit l’opérateur si le service tarde à répondre ? |

Le CSV de validation contient l’heure de production et l’heure du verdict du laboratoire. Exécutez la cellule : le verdict est-il disponible à temps pour décider à la sortie ?

In [ ]:
# Définir les budgets et mesurer le délai du laboratoire sur la période déjà connue.
import hashlib
import json
import os
import pickle
import sys
from pathlib import Path

import cjdk
import numpy as np
import pandas as pd
from IPython.display import display
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent
donnees = racine / "donnees"
val = pd.read_csv(
    donnees / "lots_validation.csv", parse_dates=["date", "label_disponible"]
)
delai_val = (val.label_disponible - val.date).dt.total_seconds() / 3600
display(delai_val.value_counts().rename("lots").rename_axis("délai en heures"))
capacite = 48
taille_fenetre = 40
places_fenetre = 8

**Votre réponse.** Pour chaque besoin, indiquez un mode de service et la mesure attendue lors d’un essai en usine.

### Calcul | Un déclenchement toutes les cinq minutes suffit-il ?

Le délai comporte au moins l’attente du prochain déclenchement et le calcul. Pour un scénario de dimensionnement, supposez un calcul et une publication de 35 secondes, sans file d’attente. Testez deux arrivées, juste après le déclenchement et une seconde avant le prochain. Recommencez pour un intervalle de deux minutes. Distinguez cette simulation d’une mesure sur Spark.

![Attente et calcul pour deux arrivées dans le même intervalle](figures/tp08-collecte.svg)

Pour la première arrivée, lisez séparément le temps d’attente et les 35 secondes de service. À quel instant faut-il commencer à mesurer le délai promis à l’opérateur ?

In [ ]:
# Simuler deux instants d’arrivée dans un intervalle de collecte.
service_secondes = 35
intervalles = [300, 120]
# Pour chaque intervalle, calculez le délai pour une arrivée à 1 et à intervalle - 1.

**À consigner.** Intervalle retenu, délai maximal simulé et marge disponible.

### Du poste de recette à un service Spark managé

Camille étudie deux déploiements possibles. Pour le pilote, une seule VM lit les fichiers et exécute le pipeline. Pour plusieurs usines, un service Spark managé peut répartir les partitions entre plusieurs machines. Le driver planifie les tâches, les exécuteurs lisent chacun une partie des données. Une jointure peut imposer un échange de données entre machines.

| Question pour l’exploitation | Spark local du TP | Spark managé |
| --- | --- | --- |
| Qui peut lire les lots et le modèle ? | Le processus de votre compte sur la VM | Tous les exécuteurs doivent accéder au stockage partagé avec une identité autorisée |
| Que devient le checkpoint si le calculateur disparaît ? | Il faut conserver le disque ou une sauvegarde exploitable | Utiliser un emplacement durable partagé, distinct des machines de calcul |
| Que mesurer avant de choisir ? | Délai sur la VM et limite de mémoire | Délai de démarrage, transferts, durée du calcul et coût par campagne |

Amazon EMR Serverless et les pools Spark Azure Synapse illustrent cette seconde architecture. [Documentation AWS](https://docs.aws.amazon.com/emr/latest/EMR-Serverless-UserGuide/emr-serverless.html) et [documentation Microsoft](https://learn.microsoft.com/en-us/azure/synapse-analytics/spark/apache-spark-overview).

**À vous.** Pour le pilote de 240 lots, quelle mesure justifierait un changement de plateforme ? Si un exécuteur distant reçoit le chemin du modèle sur votre disque personnel, que risque-t-il de se passer ? Proposez un test de reprise avant d’autoriser le passage à plusieurs sites.

**Votre réponse.** Notez votre décision et la mesure qui la justifie.

## 2. Vérifier ce qui est gelé

| Règle livrée | Entrées du modèle | Ce que vous devez retrouver |
| --- | --- | --- |
| TP 05, XGBoost local | Température, pression, durée, presse A/B | Le fichier du pipeline et son seuil |
| TP 07, Spark | Température, pression, durée | Le dossier du pipeline et le seuil de son contrat |

Une règle gelée conserve la préparation des mesures, les poids et le seuil de décision. Les cellules suivantes appliquent ces éléments aux nouveaux lots. Gardez les deux identifiants affichés pour reproduire votre bilan.

Par défaut, le notebook lit la dernière version écrite de chaque TP. Pour rejouer une livraison précise, renseignez ses deux noms dans la cellule. Utilisez seulement un pickle produit par vos propres notebooks.

In [ ]:
# Choisir les deux artefacts locaux puis contrôler les conventions avant le rejeu.
version_05 = ""  # Facultatif : nom regle-....pkl imprimé au TP 05.
version_07 = ""  # Facultatif : nom du dossier imprimé au TP 07.


def selectionner(candidats, dossier, version):
    candidats = sorted(candidats, key=lambda p: (p.stat().st_mtime_ns, p.name))
    if version:
        candidats = [p for p in candidats if p.name == version]
    if not candidats:
        raise FileNotFoundError(
            f"Aucune version utilisable dans {dossier}. Exécutez le TP correspondant ou vérifiez le nom."
        )
    return candidats[-1]


d05 = racine / "artefacts/tp05"
d07 = racine / "artefacts/tp07"
f05 = selectionner((p for p in d05.glob("regle-*.pkl") if p.is_file()), d05, version_05)
# Un export Spark complet contient le contrat et le pipeline sauvegardé.
versions_spark = (
    p
    for p in d07.glob("[!.]*")
    if (p / "contrat.json").is_file() and (p / "pipeline/metadata").is_dir()
)
f07 = selectionner(versions_spark, d07, version_07)
# Vérifier l’empreinte du fichier avant de charger le modèle local.
contenu = f05.read_bytes()
assert f05.stem == "regle-" + hashlib.sha256(contenu).hexdigest()
local = pickle.loads(contenu)
contrat = json.loads((f07 / "contrat.json").read_text())
assert local["cours"] == "MLB-TP05" and local["modèle"] == "XGBoost", (
    "Choisissez la version XGBoost du TP 05."
)
assert contrat["version"] == f07.name
assert all(
    r["comparaison"] == ">=" and r["classe_positive"] == 1 for r in (local, contrat)
)
for nom, signature in local["sources"].items():
    assert hashlib.sha256((donnees / nom).read_bytes()).hexdigest() == signature
seuils = {"Local": local["seuil"], "Spark": contrat["seuil"]}
print(f05.name, f07.name, seuils, sep="\n")

## 3. Surveiller les entrées sans attendre le laboratoire

![Médiane de température par fenêtre, validation et campagne réservée](figures/tp08-temperature.svg)

Une **fenêtre** regroupe ici 40 lots successifs. La fenêtre 1 de chaque courbe est le début de sa propre période ; les deux courbes couvrent des dates différentes. Pour chaque fenêtre, comptez les pressions absentes et calculez la médiane de température. Comparez chaque indicateur à la validation, en gardant ses unités. La fenêtre 5 a une médiane plus basse que la fenêtre 6. Quels contrôles sur le capteur ou sur les produits fabriqués permettraient d’expliquer cet écart ?

**Protocole annoncé avant le test.** Calculer les deux scores de chaque lot avec les deux versions gelées. Compter un point pour un lot conforme inspecté et dix points pour un défaut manqué. Vérifier la capacité globale de 48 inspections et, séparément, le scénario de huit créneaux par fenêtre. Un dépassement déclenche une revue de la file. Conserver les résultats initiaux dans l’audit.

In [ ]:
# Charger uniquement les entrées réservées, contrôler le schéma et la séparation temporelle.
def verifier_entrees(table):
    requis = {"id", "date", "temperature", "pression", "duree", "ligne"}
    manquantes = requis - set(table.columns)
    if manquantes:
        raise ValueError(f"Colonnes absentes : {sorted(manquantes)}")
    if table.id.isna().any() or table.id.duplicated().any():
        raise ValueError("Identifiants absents ou en double")


lots = (
    pd.read_csv(donnees / "tp08/lots.csv", parse_dates=["date"])
    .sort_values(["date", "id"])
    .reset_index(drop=True)
)
verifier_entrees(lots)
assert set(lots.id).isdisjoint(val.id) and lots.date.min() > val.label_disponible.max()
assert len(lots) == 240 and set(lots.ligne) <= {"A", "B"}
lots["fenetre"] = np.arange(len(lots)) // taille_fenetre + 1


def surveillance(table):
    return table.groupby("fenetre").agg(
        lots=("id", "size"),
        temperature_mediane=("temperature", "median"),
        pressions_absentes=("pression", lambda serie: serie.isna().sum()),
        taux_absence_pct=(
            "pression",
            lambda serie: round(100 * serie.isna().mean(), 1),
        ),
    )


display(surveillance(lots))
print(
    "Référence validation :",
    val.temperature.median(),
    "°C et",
    f"{val.pression.isna().mean():.1%}",
    "de pressions absentes",
)

### Enquête | Tester deux incidents sur une copie

Un export renomme `pression` en `pressure`. Sur un autre export, la température gagne artificiellement 5 °C. Quel contrôle doit bloquer la livraison ? Quel contrôle peut déclencher une enquête malgré un schéma correct ? Travaillez sur des copies et comparez les médianes de la même fenêtre.

In [ ]:
# Injecter les incidents sur des copies de la première fenêtre.
temoin = lots.loc[lots.fenetre.eq(1)].copy()
schema_casse = temoin.rename(columns={"pression": "pressure"})
capteur_decale = temoin.assign(temperature=temoin.temperature + 5)
# Appelez verifier_entrees dans un try/except et comparez les deux médianes.

**Décision d’exploitation.** Quel fichier isoleriez-vous ? Quel signal demanderiez-vous à la maintenance de vérifier ?

## 4. Du score du lot à la charge de l’atelier

**Atelier 8.2 du support MLB.** Sections 4 à 6.

![Score du même lot face au seuil de chacune des règles](figures/tp08-score-lot.svg)

Suivez **L0979**. Chaque point représente un score et chaque trait le seuil du modèle correspondant. Un score supérieur ou égal au seuil envoie le lot au contrôle. Quelle règle enverrait ce panneau à Camille ? Retrouvez la ligne dans le journal après exécution.

Spark vérifie d’abord si l’entrée est admissible. Un rejet technique part en vérification manuelle. Pour ce jeu de test, vérifiez que les 240 lots sont admissibles avant de comparer les deux règles sur les mêmes identifiants.

In [ ]:
# Ouvrir la session Spark et recharger la version attendue du pipeline.
os.environ["JAVA_HOME"] = str(cjdk.java_home(vendor="zulu", version="21"))
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
session = (
    SparkSession.builder.master("local[2]")
    .appName("Nervure-TP08")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "2")
)
if os.name == "nt":
    # Hadoop utilise le système de fichiers Java pour les modèles et les checkpoints.
    dossier_tp = Path.cwd()
    if dossier_tp.name == "corrections":
        dossier_tp = dossier_tp.parent
    pilote = dossier_tp / "support" / "spark" / "hadoop-bare-naked-local-fs-0.1.0.jar"
    session = (
        session.config("spark.driver.extraClassPath", str(pilote))
        .config(
            "spark.hadoop.fs.file.impl",
            "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem",
        )
        .config(
            "spark.sql.streaming.checkpointFileManagerClass",
            "org.apache.spark.sql.execution.streaming.checkpointing.FileSystemBasedCheckpointFileManager",
        )
    )
spark = session.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
assert spark.version == contrat["spark"]
m07 = PipelineModel.load(str(f07 / "pipeline"))

In [ ]:
# Transmettre une pression manquante à Spark sous forme de valeur nulle.
entrees_spark = ["id", "temperature", "pression", "duree"]
messages = lots[entrees_spark].astype(object).where(lots[entrees_spark].notna(), None)
table = spark.createDataFrame(messages.to_dict("records"), schema=contrat["schema"])
route = table.withColumn("motif", F.expr(contrat["motif_sql"]))
route.groupBy("motif").count().show()
# Le test livré doit être entièrement admissible pour comparer le coût sur le même périmètre.
assert route.where("motif != 'ok'").count() == 0
scores07 = (
    m07.transform(route)
    .select(
        "id",
        vector_to_array(F.col("probability"))[contrat["classe_positive"]].alias(
            "score_Spark"
        ),
    )
    .toPandas()
)
journal = lots.merge(scores07, on="id", how="left", validate="one_to_one")
# Aligner par identifiant avant de comparer les sorties des deux pipelines.
modele05 = local["pipeline"]
indice = list(modele05.classes_).index(local["classe_positive"])
journal["score_Local"] = modele05.predict_proba(journal[local["colonnes"]])[:, indice]
for nom, seuil in seuils.items():
    assert journal["score_" + nom].between(0, 1).all()
    journal[nom] = journal["score_" + nom].ge(seuil)
spark.stop()
display(
    journal.loc[
        journal.id.eq("L0979"), ["id", "score_Local", "Local", "score_Spark", "Spark"]
    ]
)
charge = journal.groupby("fenetre")[list(seuils)].sum()
display(charge)
print("Inspections totales :", journal[list(seuils)].sum().to_dict())

### Arbitrage | Combien de places manqueraient localement ?

![Inspections par fenêtre et limite de huit places](figures/tp08-charge.svg)

Repérez la première fenêtre qui dépasse huit inspections pour chaque modèle. Si les créneaux ne peuvent pas passer d’une fenêtre à l’autre, additionnez **uniquement** les excédents. Expliquez ce qu’une comparaison globale à 48 masque.

In [ ]:
# Isoler les fenêtres saturées pour préparer un plan de renfort.
ecarts_charge = charge - places_fenetre
# Utilisez clip(lower=0) avant de sommer.

**À consigner.** Places à ajouter au total pour chaque règle si les créneaux ne sont pas transférables.

## 5. À partir de quand peut-on juger la décision ?

Un lot en attente de verdict compte déjà dans la charge de travail. Il entre dans le calcul des défauts détectés ou manqués une fois le résultat du laboratoire disponible.

Le fichier des décisions reste inchangé. Ouvrez maintenant les verdicts et rattachez chacun à son lot par identifiant. À l’heure de sortie du dernier panneau, combien de décisions pouvez-vous compter ? Combien pouvez-vous déjà évaluer ? Refaites le calcul à l’heure du dernier retour.

In [ ]:
# Ajouter les étiquettes différées seulement après la fixation des décisions.
verdicts = pd.read_csv(donnees / "tp08/verdicts.csv", parse_dates=["label_disponible"])
assert verdicts.id.is_unique and set(verdicts.id) == set(journal.id)
audit = journal.merge(verdicts, on="id", validate="one_to_one")
assert audit.defaut.isin([0, 1]).all() and audit.label_disponible.ge(audit.date).all()
for instant in (audit.date.max(), audit.label_disponible.max()):
    verdicts_recus = audit.label_disponible.le(instant)
    print(instant, "verdicts reçus :", int(verdicts_recus.sum()), "/", len(audit))
assert audit.label_disponible.notna().all()

**À vous.** Indiquez la date de clôture du bilan, le dénominateur aux deux instants et le traitement du dernier lot en attente.

## 6. Que gagne-t-on avec ces inspections ?

![Lots défectueux inspectés et manqués parmi les 75 défauts confirmés](figures/tp08-rappel.svg)

**1. Lire le rappel.** Parmi les lots réellement défectueux, un vrai positif (VP) a été envoyé au contrôle. Un faux négatif (FN) a échappé au contrôle. Retrouvez le dénominateur en additionnant les points pleins et vides pour chaque modèle. Les points sont regroupés par décision. Pour comparer des identifiants précis, utilisez le journal du premier défi.

$$
\mathrm{rappel}=\frac{VP}{VP+FN}
$$

**2. Calculer la charge et le coût.** Un faux positif (FP) est un lot conforme envoyé au contrôle. Il occupe une place, tout comme un défaut intercepté. Un vrai négatif (VN) est un lot conforme resté en file ordinaire.

$$
\mathrm{inspections}=VP+FP
$$

$$
C=FP+10\,FN
$$

Reconstituez les trois indicateurs du modèle local. La référence « aucune inspection » laisse passer tous les défauts. Combien de points la règle économise-t-elle par rapport à cette référence ?

Lisez le coût en points. Prenons deux règles fictives appliquées aux mêmes lots. La première manque trois défauts et lance six fausses alertes, soit 36 points. La seconde manque deux défauts et lance neuf fausses alertes, soit 29 points. Le deuxième bilan économise sept points, mais consomme quatre inspections de plus (un défaut intercepté et trois fausses alertes supplémentaires). Avant de le recommander, vérifiez que l’atelier peut absorber ces quatre contrôles.

In [ ]:
# Évaluer toutes les décisions sur les mêmes lots, avec la même convention.
def bilan(decisions, defauts):
    positifs = defauts.eq(1)
    vp = int((decisions & positifs).sum())
    fp = int((decisions & ~positifs).sum())
    fn = int((~decisions & positifs).sum())
    vn = len(decisions) - vp - fp - fn
    return {
        "VP": vp,
        "FP": fp,
        "FN": fn,
        "VN": vn,
        "inspections": vp + fp,
        "rappel": vp / (vp + fn) if vp + fn else np.nan,
        "cout": fp + 10 * fn,
    }


resultats = pd.DataFrame({nom: bilan(audit[nom], audit.defaut) for nom in seuils}).T
sans_inspection = 10 * int(audit.defaut.sum())
resultats["gain"] = sans_inspection - resultats.cout
display(resultats.round(3))
assert resultats[["VP", "FP", "FN", "VN"]].sum(axis=1).eq(len(audit)).all()

In [ ]:
# Rattacher chaque chiffre du bilan au fichier et au seuil effectivement utilisés.
fiche = resultats.assign(
    version=pd.Series({"Local": f05.name, "Spark": f07.name}),
    seuil=pd.Series(seuils),
    depassement_global=(resultats.inspections - capacite).clip(lower=0),
    renfort_par_fenetre=ecarts_charge.clip(lower=0).sum(),
)
display(fiche)
print("Clôture de l’expertise :", audit.label_disponible.max())

### Fiche de décision à remettre à Camille

![Charge et coût des deux règles face aux 48 places disponibles](figures/tp08-arbitrage.svg)

Quelle règle respecterait le plafond sans intervention de Camille ? Appuyez votre avis sur le coût **et** sur la capacité.

Complétez cette fiche avec vos propres valeurs.

| Décision à documenter | Votre réponse |
| --- | --- |
| Besoin métier, échéance et architecture proposée | |
| Versions exactes du TP 05 et du TP 07, seuils appliqués | |
| Période finale, lots admis, rejets et date de clôture | |
| Charge globale et fenêtres où il manque des places | |
| FP, FN, rappel, coût et gain pour chaque règle | |
| Mise en service, pilote limité ou maintien en observation : pourquoi ? | |
| Indicateur immédiat, indicateur différé, responsable et action | |
| Critère de suspension et procédure de retour à une version approuvée | |

## Défi facultatif | Les deux modèles manquent-ils les mêmes défauts ?

![Défauts interceptés selon les quatre combinaisons de décisions](figures/tp08-desaccords.svg)

Formez quatre groupes selon la décision locale et la décision Spark. Comptez les lots et les défauts de chaque groupe. Camille propose de retenir tout lot signalé par au moins un modèle. Chiffrez cette hypothèse à titre de diagnostic. Cette troisième règle pourrait-elle être approuvée sur la seule base de ce calcul ?

In [ ]:
# Apparier les décisions pour étudier une proposition de cumul.
groupes = audit.groupby(["Local", "Spark"]).defaut.agg(["size", "sum"])
display(groupes)
# Utilisez le OU entre les deux colonnes et appliquez bilan.

**Arbitrage.** Combien de défauts supplémentaires et de places supplémentaires par rapport à XGBoost seul ?

## Défi facultatif | Comparer les presses A et B

![Rappels et effectifs par presse](figures/tp08-presses.svg)

Un même nombre de défauts manqués n’a pas le même sens sur une presse qui produit dix défauts et sur une presse qui en produit cinquante. Pour chaque presse et chaque modèle, calculez le rappel avec son numérateur et son dénominateur. Quel groupe mérite une enquête ? Quels facteurs pourraient expliquer un écart ?

In [ ]:
# Comparer les décisions au sein de chaque presse en gardant les effectifs.
for ligne, groupe in audit.groupby("ligne"):
    print(ligne, "lots :", len(groupe), "défauts :", groupe.defaut.sum())
    # Appelez bilan pour chacun des deux modèles.

**À consigner.** Indiquez la presse et le modèle à examiner, avec le nombre de défauts observés.

## Défi facultatif | Un verdict qui arrive en retard

![Évolution de la couverture avec les dates de retour initiales et simulées](figures/tp08-retard.svg)

Simulez un retard de 24 heures sur les retours de la dernière fenêtre. Rejouez la clôture initiale, puis la clôture différée. Quels lots peuvent encore alimenter le suivi de charge ? Lesquels seulement peuvent alimenter le rappel ? Vérifiez que le rapport définitif revient aux valeurs originales une fois tous les verdicts reçus.

In [ ]:
# Simuler un incident du laboratoire sur une copie, décisions inchangées.
retard = audit.copy()
retard.loc[retard.fenetre.eq(retard.fenetre.max()), "label_disponible"] += pd.Timedelta(
    hours=24
)
cloture_initiale = audit.label_disponible.max()
# Filtrez les verdicts reçus à cette date, puis à la nouvelle clôture.

**À consigner.** Combien de lots ont une décision, puis combien ont un verdict à chaque clôture ?

[Ouvrir le corrigé expliqué](corrections/TP08_architecture_surveillance_et_evaluation.ipynb)